# March Madness (Women's) - Training Pipeline

Run this notebook once per season to calibrate `ModelParams` from historical NCAA Women's tournament outcomes and save them to `data/W_trained_params.json` for the simulator notebook.

## Data required

Download the Kaggle March Mania dataset and place these files in `data/kaggle/`:
- `WNCAATourneyCompactResults.csv`
- `WNCAATourneySeeds.csv`
- `WTeams.csv`
- `WRegularSeasonDetailedResults.csv` (for season stats)

No KenPom login required — all data comes from Kaggle files.

## Runtime
| Step | Time |
|------|------|
| Load Kaggle season stats | ~5 s |
| Build historical schedule features | ~15 s |
| Build training dataset | ~10 s |
| Train params (7 shock_df values) | ~1-2 min |
| Cross-validation | ~15-20 min |

If you want a quick pass, skip Cell 3 and train without the schedule-derived features.

In [1]:
# Cell 1 - Imports
import os
import sys

sys.path.insert(0, ".")

from marchmadness import (
    load_kaggle_season_stats,
    build_kaggle_schedule_features,
    load_kaggle_historical,
    build_training_dataset,
    fit_feature_scaler,
    apply_feature_scaler,
    calibrate_shock_params,
    ModelParams,
    train_params,
    cross_validate,
    evaluate_params,
    predict_probs,
    save_params,
    save_scaler,
    HISTORICAL_SEED_WIN_RATES,
)

In [2]:
# Cell 2 - Load historical women's data from Kaggle
#
# No browser/login required. All data comes from Kaggle CSV files.
# Stats are loaded from WRegularSeasonDetailedResults.csv.

TRAIN_YEARS = list(range(2010, 2026))

kenpom_stats = load_kaggle_season_stats(TRAIN_YEARS, "data/kaggle", gender="W")
print(f"Loaded stats for {len(kenpom_stats)} seasons: {sorted(kenpom_stats.keys())}")

kaggle_df = load_kaggle_historical(
    "data/kaggle/WNCAATourneyCompactResults.csv",
    "data/kaggle/WNCAATourneySeeds.csv",
)
print(
    f"Loaded {len(kaggle_df)} Kaggle tournament games "
    f"({kaggle_df['Season'].min()}-{kaggle_df['Season'].max()})"
)

Loaded stats for 16 seasons: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Loaded 1709 Kaggle tournament games (1998-2025)


In [3]:
# Cell 3 - Build historical schedule features  [OPTIONAL]
#
# Uses Kaggle regular-season results to derive:
#   recent_form        -> opponent-weighted win rate over the last 10 games
#   season_trajectory  -> trend in cumulative weighted performance over the full season
#   conference tournament wins
#
# Uses build_kaggle_schedule_features instead of build_historical_schedule_context
# (no KenPom schedules required).

historical_schedule_features, historical_conf_tourney = build_kaggle_schedule_features(
    TRAIN_YEARS, "data/kaggle", gender="W", recent_window=10
)

team_seasons = sum(len(v) for v in historical_schedule_features.values())
print(
    f"Built schedule features for {team_seasons} team-seasons across "
    f"{len(historical_schedule_features)} years."
)

# To skip these features entirely, comment out the block above and uncomment:
# historical_schedule_features = None
# historical_conf_tourney = None

Built schedule features for 5603 team-seasons across 16 years.


In [4]:
# Cell 4 - Build training dataset
#
# Joins Kaggle NCAA Women's tournament outcomes with season stats and schedule features.
# WMasseyOrdinals.csv may not have the same structure as men's, so kaggle_features_df
# is set to None. Add extra name fixes if the build step warns about unresolved teams.

extra_name_map = {
    # Example: "App State": "Appalachian St."
}

training_df = build_training_dataset(
    kaggle_df,
    kenpom_stats,
    kaggle_teams_path="data/kaggle/WTeams.csv",
    name_map=extra_name_map,
    schedule_features=historical_schedule_features,
    conf_tourney_results=historical_conf_tourney,
    kaggle_features_df=None,   # WMasseyOrdinals may differ from men's structure
    min_year=2010,
)

print(f"Dataset shape: {training_df.shape}")
print(f"Label balance: {training_df['label'].mean():.3f}  (expect ~0.50)")

training_df.head()
# Fit feature scaler on the full training dataset.
feature_scaler = fit_feature_scaler(training_df)
training_df_scaled = apply_feature_scaler(training_df, feature_scaler)

print("Feature scaler stats:")
for feat, stats in feature_scaler.items():
    print(f"  {feat}: mean={stats['mean']:.4f}  std={stats['std']:.4f}")

Training dataset: 953 games | 15 seasons (2010-2025) | schedule coverage: 100% of team-slots
Dataset shape: (953, 47)
Label balance: 0.503  (expect ~0.50)
Feature scaler stats:
  luck: mean=-0.1442  std=0.1502
  recent_form: mean=-0.0671  std=0.1523
  season_trajectory: mean=0.0000  std=1.0000
  neutral_win_pct: mean=0.5000  std=1.0000
  coach_tourney_wins: mean=0.0000  std=1.0000
  efg_o: mean=49.3915  std=3.7464
  efg_d: mean=42.1559  std=2.9992
  tov_o: mean=20.1708  std=2.6247
  tov_d: mean=23.3087  std=3.9212
  orb: mean=35.5784  std=5.2929
  drb: mean=71.0238  std=4.1576
  ftr_o: mean=30.8031  std=4.9813
  ftr_d: mean=27.3889  std=5.4937


In [5]:
# Cell 5 - Calibrate shock parameters
#
# Finds (shock_df, shock_scale) that reproduce historical seed-matchup upset rates.
# These are fixed before feature-weight training so the optimizer can't collapse
# noise to zero in pursuit of lower log-loss.
#
# Strategy: simulate games between teams with identical stats (zero
# base_margin). Only the shock terms contribute, so the resulting upset rate
# purely reflects the shock distribution shape.

best_margin_scale, best_shock_df, best_shock_scale = calibrate_shock_params(training_df_scaled)

# Bake into params_init so train_params inherits them as fixed
params = ModelParams(
    margin_scale=best_margin_scale,
    shock_df=best_shock_df,
    shock_scale=best_shock_scale,
)
print(f"Calibrated: margin_scale={params.margin_scale:.2f}  shock_df={params.shock_df:.2f}  shock_scale={params.shock_scale:.4f}")

Calibration complete: margin_scale=70.01, shock_df=5.01, shock_scale=0.0769  (SSE=0.007383)
  Seed matchup fit:
     1v16  fit=0.963  target=0.990
     2v15  fit=0.917  target=0.941
     3v14  fit=0.846  target=0.848
     4v13  fit=0.778  target=0.791
     5v12  fit=0.721  target=0.648
     6v11  fit=0.616  target=0.630
     7v10  fit=0.580  target=0.598
     8v9   fit=0.517  target=0.513
Calibrated: margin_scale=70.01  shock_df=5.01  shock_scale=0.0769


In [6]:
# Cell 6 - Train parameters
#
# Strategy: outer grid search over shock_df, inner L-BFGS-B over
# margin_scale, shock_scale, luck_weight, recent_form_weight,
# trajectory_weight, and seed_prior_weight.
#
# Note: the metrics printed below are in-sample because they evaluate on the
# same training_df used for fitting. Use Cell 6 cross-validation to judge
# whether the feature set actually generalizes.

best_params = train_params(
    training_df_scaled,
    params_init=params,
    verbose=True,
)

metrics = evaluate_params(training_df_scaled, best_params)
print("In-sample metrics:")
print(f"  log_loss:    {metrics['log_loss']:.4f}  (random baseline = 0.6931)")
print(f"  brier_score: {metrics['brier_score']:.4f}")
print(f"  accuracy:    {metrics['accuracy']:.3f}")
print(f"  better-seed baseline accuracy: {metrics['seed_baseline_accuracy']:.3f}")
print(f"  higher-AdjEM baseline accuracy: {metrics['adj_em_baseline_accuracy']:.3f}")
print(f"  mean confidence: {metrics['mean_confidence']:.3f}")
print(
    f"  high-confidence accuracy (>=70%): {metrics['high_confidence_accuracy']:.3f} "
    f"on {metrics['high_confidence_share']:.1%} of games"
)

print("Simulated upset rates vs historical:")
for (lo, hi), hist_rate in sorted(HISTORICAL_SEED_WIN_RATES.items()):
    sim = metrics['upset_rates'].get((lo, hi), float('nan'))
    print(f"  {lo:2d}v{hi:<2d}  sim={sim:.3f}  hist={1-hist_rate:.3f}")

log-loss: 0.416837  (random baseline = 0.6931)
shock_df=5.01  shock_scale=0.0769  luck_w=0.0000  recent_form_w=0.0000  w_efg=0.0140  w_to=-0.0020  w_orb=0.0156  w_ftr=-0.0082
In-sample metrics:
  log_loss:    0.4168  (random baseline = 0.6931)
  brier_score: 0.1375
  accuracy:    0.797
  better-seed baseline accuracy: 0.784
  higher-AdjEM baseline accuracy: 0.806
  mean confidence: 0.798
  high-confidence accuracy (>=70%): 0.884 on 69.5% of games
Simulated upset rates vs historical:
   1v16  sim=0.000  hist=0.010
   2v15  sim=0.000  hist=0.059
   3v14  sim=0.000  hist=0.152
   4v13  sim=0.033  hist=0.209
   5v12  sim=0.200  hist=0.352
   6v11  sim=0.317  hist=0.370
   7v10  sim=0.350  hist=0.402
   8v9   sim=0.433  hist=0.487


In [7]:
# Cell 7 - Cross-validation  [OPTIONAL]
#
# Leave-one-year-out validation. Use a smaller shock_df_grid for a quick check.

cv_results = cross_validate(
    training_df_scaled,
    params_init=params,
    verbose=True,
)

print("CV results:")
print(cv_results[[
    "year", "n_val", "log_loss", "brier_score", "accuracy",
    "luck_weight", "recent_form_weight",
]].to_string(index=False))


--- Fold: val=2010  train=2011-2025 (890 train, 63 val) ---
log-loss: 0.416736  (random baseline = 0.6931)
shock_df=5.01  shock_scale=0.0769  luck_w=0.0000  recent_form_w=0.0000  w_efg=0.0164  w_to=0.0017  w_orb=0.0144  w_ftr=-0.0013
  -> val log_loss=0.4207  brier=0.1411  acc=0.810

--- Fold: val=2011  train=2010-2025 (890 train, 63 val) ---
log-loss: 0.418658  (random baseline = 0.6931)
shock_df=5.01  shock_scale=0.0769  luck_w=0.0000  recent_form_w=0.0000  w_efg=0.0102  w_to=-0.0005  w_orb=0.0157  w_ftr=-0.0093
  -> val log_loss=0.3921  brier=0.1322  acc=0.762

--- Fold: val=2012  train=2010-2025 (890 train, 63 val) ---
log-loss: 0.419756  (random baseline = 0.6931)
shock_df=5.01  shock_scale=0.0769  luck_w=0.0000  recent_form_w=0.0000  w_efg=0.0128  w_to=-0.0062  w_orb=0.0156  w_ftr=-0.0103
  -> val log_loss=0.3765  brier=0.1172  acc=0.873

--- Fold: val=2013  train=2010-2025 (890 train, 63 val) ---
log-loss: 0.414584  (random baseline = 0.6931)
shock_df=5.01  shock_scale=0.0769  

In [8]:
# Cell 7b - Calibration table
#
# Bucket predictions into 5% bins and compare mean predicted probability
# against actual win rate. A well-calibrated model lands on the diagonal.
# Systematic over/under-confidence shows as buckets above/below diagonal.
#
# NOTE: this uses in-sample predictions — same data that trained the model.
# Cross-validation calibration (using held-out years) is more rigorous but
# requires iterating over CV folds. This in-sample view is sufficient to spot
# gross miscalibration (e.g. 80%-confident predictions winning only 65%).

import numpy as np

probs = predict_probs(best_params, training_df_scaled)
labels = training_df_scaled["label"].to_numpy(dtype=float)

print("Calibration table (in-sample)")
print(f"{'Bucket':>12}  {'Pred':>6}  {'Actual':>6}  {'N':>5}  {'Gap':>6}")
print("-" * 46)
for lo in np.arange(0.0, 1.0, 0.05):
    hi = lo + 0.05
    mask = (probs >= lo) & (probs < hi)
    n = mask.sum()
    if n == 0:
        continue
    bucket_pred   = probs[mask].mean()
    bucket_actual = labels[mask].mean()
    gap = bucket_pred - bucket_actual
    flag = "  <-- overconfident" if gap > 0.07 else ("  <-- underconfident" if gap < -0.07 else "")
    print(f"{lo:5.0%}\u2013{hi:5.0%}  {bucket_pred:6.1%}  {bucket_actual:6.1%}  {n:5d}  {gap:+6.1%}{flag}")

Calibration table (in-sample)
      Bucket    Pred  Actual      N     Gap
----------------------------------------------
   0%–   5%    2.0%    0.0%    111   +2.0%
   5%–  10%    7.2%   11.3%     62   -4.0%
  10%–  15%   12.5%   17.0%     53   -4.5%
  15%–  20%   17.7%   13.9%     36   +3.8%
  20%–  25%   22.3%   36.8%     38  -14.5%  <-- underconfident
  25%–  30%   27.7%   25.0%     36   +2.7%
  30%–  35%   32.3%   31.4%     35   +0.8%
  35%–  40%   37.6%   39.1%     46   -1.5%
  40%–  45%   42.4%   43.9%     41   -1.5%
  45%–  50%   47.3%   44.2%     43   +3.1%
  50%–  55%   52.5%   55.2%     29   -2.6%
  55%–  60%   57.8%   50.0%     28   +7.8%  <-- overconfident
  60%–  65%   62.8%   54.1%     37   +8.7%  <-- overconfident
  65%–  70%   67.6%   81.2%     32  -13.7%  <-- underconfident
  70%–  75%   72.3%   78.6%     42   -6.2%
  75%–  80%   77.7%   80.0%     35   -2.3%
  80%–  85%   82.7%   80.0%     35   +2.7%
  85%–  90%   87.6%   90.2%     51   -2.6%
  90%–  95%   92.4%   91.9%

In [9]:
# Cell 7c - Round-conditional calibration
#
# Shows calibration separately for each tournament round.
# Requires re-running Cell 4 (build_training_dataset) after the training.py
# update that adds day_num to each row.
#
# DayNum round mapping (Kaggle convention):
#   134-135 → First Four
#   136-137 → Round of 64
#   138-140 → Round of 32
#   143-144 → Sweet 16
#   145-148 → Elite 8
#   152     → Final Four
#   154     → Championship

def _day_to_round(day: int) -> str:
    if day <= 135:  return "First Four"
    if day <= 137:  return "Round of 64"
    if day <= 140:  return "Round of 32"
    if day <= 144:  return "Sweet 16"
    if day <= 148:  return "Elite 8"
    if day <= 152:  return "Final Four"
    return "Championship"

if "day_num" not in training_df_scaled.columns:
    print("day_num column not found — re-run Cell 4 (build_training_dataset) to populate it.")
else:
    round_labels_col = training_df_scaled["day_num"].apply(_day_to_round)
    round_order = ["Round of 64", "Round of 32", "Sweet 16", "Elite 8", "Final Four", "Championship"]

    print("Round-conditional calibration (in-sample)")
    print(f"{'Round':<18}  {'N':>5}  {'Pred':>6}  {'Actual':>6}  {'Acc':>5}  {'Gap':>6}")
    print("-" * 58)
    for rnd in round_order:
        mask = round_labels_col == rnd
        if not mask.any():
            continue
        p = probs[mask.to_numpy()]
        y = labels[mask.to_numpy()]
        n = mask.sum()
        mean_pred   = p.mean()
        mean_actual = y.mean()
        acc = ((p >= 0.5) == y).mean()
        gap = mean_pred - mean_actual
        flag = "  **" if abs(gap) > 0.07 else ""
        print(f"{rnd:<18}  {n:5d}  {mean_pred:6.1%}  {mean_actual:6.1%}  {acc:5.1%}  {gap:+6.1%}{flag}")

    # Detail: calibration buckets within Sweet 16+ only
    late_mask = round_labels_col.isin(["Sweet 16", "Elite 8", "Final Four", "Championship"])
    if late_mask.any():
        p_late = probs[late_mask.to_numpy()]
        y_late = labels[late_mask.to_numpy()]
        print(f"\nCalibration buckets — Sweet 16 and later (n={late_mask.sum()})")
        print(f"{'Bucket':>12}  {'Pred':>6}  {'Actual':>6}  {'N':>5}  {'Gap':>6}")
        print("-" * 46)
        for lo in np.arange(0.0, 1.0, 0.10):
            hi = lo + 0.10
            m = (p_late >= lo) & (p_late < hi)
            n = m.sum()
            if n == 0:
                continue
            bucket_pred   = p_late[m].mean()
            bucket_actual = y_late[m].mean()
            gap = bucket_pred - bucket_actual
            flag = "  <-- overconfident" if gap > 0.07 else ("  <-- underconfident" if gap < -0.07 else "")
            print(f"{lo:5.0%}\u2013{hi:5.0%}  {bucket_pred:6.1%}  {bucket_actual:6.1%}  {n:5d}  {gap:+6.1%}{flag}")

Round-conditional calibration (in-sample)
Round                   N    Pred  Actual    Acc     Gap
----------------------------------------------------------
Round of 64           152   46.3%   47.4%  77.6%   -1.0%
Round of 32           520   50.6%   51.9%  82.1%   -1.3%
Sweet 16               92   46.2%   45.7%  72.8%   +0.6%
Elite 8               144   47.7%   49.3%  80.6%   -1.6%
Final Four             16   53.3%   50.0%  56.2%   +3.3%
Championship           29   50.6%   55.2%  79.3%   -4.6%

Calibration buckets — Sweet 16 and later (n=281)
      Bucket    Pred  Actual      N     Gap
----------------------------------------------
   0%–  10%    5.3%    7.5%     40   -2.2%
  10%–  20%   14.9%   16.2%     37   -1.4%
  20%–  30%   25.1%   30.8%     26   -5.7%
  30%–  40%   35.8%   39.4%     33   -3.6%
  40%–  50%   44.7%   45.0%     20   -0.3%
  50%–  60%   54.9%   53.8%     13   +1.1%
  60%–  70%   64.7%   71.4%     14   -6.7%
  70%–  80%   74.1%   70.0%     30   +4.1%
  80%–  90%   8

In [12]:
# Cell 7f - Temperature scaling + isotonic calibration
#
# Two post-hoc calibration approaches applied on top of the trained model.
# Neither changes any learned weights — they only remap raw probabilities.
#
# Temperature scaling:
#   p_cal = sigmoid(logit / tau),  tau > 1 softens, tau < 1 sharpens.
#   tau is fitted by minimizing log-loss with a single scalar.
#
# Isotonic calibration:
#   sklearn IsotonicRegression fits a non-parametric monotone map p -> p_cal.

from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression

# --- helpers ---

def _metrics(p: np.ndarray, y: np.ndarray) -> dict:
    p = np.clip(p, 1e-7, 1 - 1e-7)
    ll  = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    bs  = float(np.mean((p - y) ** 2))
    acc = float(np.mean((p >= 0.5) == y))
    # ECE: weighted mean |gap| across 10 equal-width bins
    ece = 0.0
    for lo in np.arange(0, 1, 0.10):
        m = (p >= lo) & (p < lo + 0.10)
        if m.sum() == 0: continue
        ece += m.sum() * abs(p[m].mean() - y[m].mean())
    ece /= len(p)
    return {"log_loss": ll, "brier": bs, "acc": acc, "ece": ece}

# --- temperature scaling ---
def _tau_logloss(tau, logits, y):
    p = 1 / (1 + np.exp(-np.clip(logits / tau, -500, 500)))
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

result = minimize_scalar(_tau_logloss, bounds=(0.5, 3.0), method="bounded",
                         args=(logits, labels_arr))
best_tau = float(result.x)
print(f"Temperature scaling: optimal tau = {best_tau:.4f}  (tau>1 softens)")

p_temp = np.clip(1 / (1 + np.exp(-logits / best_tau)), 1e-7, 1 - 1e-7)

# --- isotonic calibration ---
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(probs, labels_arr)
p_iso = np.clip(iso.predict(probs), 1e-7, 1 - 1e-7)

# --- comparison table ---
configs = [
    ("Raw model",            probs),
    (f"Temperature (\u03c4={best_tau:.2f})", p_temp),
    ("Isotonic",             p_iso),
]

print()
print(f"{'':30}  {'LogLoss':>7}  {'Brier':>7}  {'Acc':>5}  {'ECE':>5}  80-85%  85-90%  90-95%  95-100%")
print("-" * 100)
for name, p in configs:
    m = _metrics(p, labels_arr)
    parts = []
    for lo in [0.80, 0.85, 0.90, 0.95]:
        hi = lo + 0.05
        mask = (p >= lo) & (p < hi)
        if mask.sum() < 3: parts.append("  n/a "); continue
        gap = p[mask].mean() - labels_arr[mask].mean()
        parts.append(f"{gap:+5.1%}{'*' if abs(gap)>0.07 else ' '}")
    print(f"  {name:<28}  {m['log_loss']:7.4f}  {m['brier']:7.4f}  {m['acc']:5.3f}  {m['ece']:5.4f}  {'  '.join(parts)}")

Temperature scaling: optimal tau = 0.9743  (tau>1 softens)

                                LogLoss    Brier    Acc    ECE  80-85%  85-90%  90-95%  95-100%
----------------------------------------------------------------------------------------------------
  Raw model                      0.4168   0.1375  0.797  0.0159  +2.7%   -2.6%   +0.5%   -2.1% 
  Temperature (τ=0.97)           0.4168   0.1375  0.797  0.0185  +1.9%   -3.0%   +1.9%   -2.0% 
  Isotonic                       0.3964   0.1327  0.800  0.0000    n/a   -0.0%     n/a   -0.0% 


In [13]:
# Cell 7g - Cross-validate temperature tau
#
# Fit tau on each held-out year independently to check stability.
# If tau stays in a narrow band across folds (e.g. 1.4-1.7), it is a real
# structural correction and safe to apply globally.
#
# Final tau: median of per-fold estimates (robust to outlier years).

import dataclasses

years = sorted(training_df_scaled["year"].unique())
tau_estimates = []

print(f"{'Year':>6}  {'tau':>6}  {'raw_ll':>8}  {'cal_ll':>8}  {'delta_ll':>9}")
print("-" * 46)

for val_year in years:
    train_mask = training_df_scaled["year"] != val_year
    val_mask   = training_df_scaled["year"] == val_year

    df_val = training_df_scaled[val_mask]
    y_val  = df_val["label"].to_numpy(dtype=float)

    # Fit tau on training folds (not val), using best_params as base
    df_train = training_df_scaled[train_mask]
    y_train  = df_train["label"].to_numpy(dtype=float)

    # Raw logits on training data (tau=1)
    p_train_raw = predict_probs(dataclasses.replace(best_params, temperature=1.0), df_train)
    logits_train = np.log(np.clip(p_train_raw, 1e-6, 1-1e-6) / (1 - np.clip(p_train_raw, 1e-6, 1-1e-6)))

    def _tau_ll_train(tau):
        p = np.clip(1 / (1 + np.exp(-logits_train / tau)), 1e-9, 1-1e-9)
        return -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))

    res = minimize_scalar(_tau_ll_train, bounds=(0.5, 3.0), method="bounded")
    fold_tau = float(res.x)
    tau_estimates.append(fold_tau)

    # Evaluate on val year: raw vs calibrated
    p_val_raw = predict_probs(dataclasses.replace(best_params, temperature=1.0), df_val)
    logits_val = np.log(np.clip(p_val_raw, 1e-6, 1-1e-6) / (1 - np.clip(p_val_raw, 1e-6, 1-1e-6)))
    p_val_cal = np.clip(1 / (1 + np.exp(-logits_val / fold_tau)), 1e-9, 1-1e-9)

    ll_raw = -np.mean(y_val * np.log(np.clip(p_val_raw,1e-9,1-1e-9)) + (1-y_val)*np.log(np.clip(1-p_val_raw,1e-9,1-1e-9)))
    ll_cal = -np.mean(y_val * np.log(p_val_cal) + (1-y_val)*np.log(1-p_val_cal))
    print(f"{val_year:>6}  {fold_tau:>6.3f}  {ll_raw:>8.4f}  {ll_cal:>8.4f}  {ll_cal-ll_raw:>+9.4f}")

tau_median = float(np.median(tau_estimates))
tau_mean   = float(np.mean(tau_estimates))
tau_std    = float(np.std(tau_estimates))
print(f"\n\u03c4 estimates:  mean={tau_mean:.3f}  median={tau_median:.3f}  std={tau_std:.3f}")
print(f"Range: [{min(tau_estimates):.3f}, {max(tau_estimates):.3f}]")
print(f"\nUsing \u03c4 = {tau_median:.4f} (median across folds)")

  Year     tau    raw_ll    cal_ll   delta_ll
----------------------------------------------
  2010   0.976    0.4160    0.4157    -0.0002
  2011   0.981    0.3902    0.3896    -0.0005
  2012   0.989    0.3748    0.3742    -0.0006
  2013   0.968    0.4454    0.4461    +0.0007
  2014   0.970    0.4243    0.4247    +0.0004
  2015   0.985    0.3731    0.3724    -0.0006
  2016   0.963    0.4518    0.4533    +0.0015
  2017   0.964    0.4355    0.4367    +0.0012
  2018   0.942    0.4940    0.5009    +0.0069
  2019   0.986    0.3680    0.3674    -0.0006
  2021   0.965    0.4372    0.4382    +0.0010
  2022   0.969    0.4584    0.4588    +0.0004
  2023   0.954    0.4946    0.4979    +0.0033
  2024   0.995    0.3660    0.3657    -0.0004
  2025   1.008    0.3241    0.3250    +0.0010

τ estimates:  mean=0.974  median=0.970  std=0.016
Range: [0.942, 1.008]

Using τ = 0.9698 (median across folds)


In [14]:
# Cell 8 - Save trained params
import os
import dataclasses

os.makedirs("data", exist_ok=True)

# Bake the cross-validated temperature into best_params before saving.
# tau_median is set by Cell 7g; fall back to the in-sample tau if that cell
# was not run (e.g. quick pass skipping CV).
try:
    final_tau = tau_median
except NameError:
    final_tau = best_tau   # from Cell 7f in-sample fit
    print(f"Warning: tau_median not found (Cell 7g skipped). Using in-sample tau={final_tau:.4f}.")

best_params = dataclasses.replace(best_params, temperature=final_tau)

save_params(best_params, "data/W_trained_params.json")

print("Trained params summary:")
print(f"  margin_scale           = {best_params.margin_scale:.2f}")
print(f"  shock_df               = {best_params.shock_df:.2f}")
print(f"  shock_scale            = {best_params.shock_scale:.4f}")
print(f"  luck_weight            = {best_params.luck_weight:.4f}")
print(f"  recent_form_weight     = {best_params.recent_form_weight:.4f}")
print(f"  seed_prior_weight      = {best_params.seed_prior_weight:.4f}")
print(f"  w_efg                  = {best_params.w_efg:.4f}")
print(f"  w_to                   = {best_params.w_to:.4f}")
print(f"  w_orb                  = {best_params.w_orb:.4f}")
print(f"  w_ftr                  = {best_params.w_ftr:.4f}")
print(f"  temperature (tau)      = {best_params.temperature:.4f}")

save_scaler(feature_scaler, "data/W_feature_scaler.json")
print("Feature scaler saved to data/W_feature_scaler.json")

Params saved to data/W_trained_params.json
Trained params summary:
  margin_scale           = 70.01
  shock_df               = 5.01
  shock_scale            = 0.0769
  luck_weight            = 0.0000
  recent_form_weight     = 0.0000
  seed_prior_weight      = 0.2091
  w_efg                  = 0.0140
  w_to                   = -0.0020
  w_orb                  = 0.0156
  w_ftr                  = -0.0082
  temperature (tau)      = 0.9698
Scaler saved to data/W_feature_scaler.json
Feature scaler saved to data/W_feature_scaler.json
